In [1]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
 
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)
from sklearn.model_selection import cross_val_score



In [2]:
#Set the emojis 

POSITIVE_EMOJIS  = set('🥳🤩🔥😁😍✨💯🎉👍🌟😊')
NEUTRAL_EMOJIS   = set('🫤🙂🤷😐')
NEGATIVE_EMOJIS  = set('😒😠😞💔👎😤😢')
SARCASTIC_EMOJIS = set('🙄👏😂🙃')
 
SOCIAL_SLANG = {
    'tbh': 'to be honest', 'ngl': 'not gonna lie', 'omg': 'oh my god',
    'idk': 'i do not know', 'fr':  'for real',      'highkey': 'really',
    'lowkey': 'slightly',   'lit': 'amazing',        'goat': 'best ever',
    'sus':  'suspicious',   'slay': 'excellent',     'based': 'good',
    'cap':  'lie',          'no cap': 'truth',       'bussin': 'great food',
    'vibe': 'feeling',
}

In [3]:
#Emoji processor

def extract_emojis(text):
    """Return list of emojis found in text."""
    emoji_pattern = re.compile(
        "[\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F9FF"
        "\U00002702-\U000027B0"
        "\U0001FA00-\U0001FA9F]+",
        flags=re.UNICODE
    )
    return emoji_pattern.findall(text)
 
def emoji_to_sentiment_tag(text):
    """Replace emojis with sentiment tags that NB can learn from."""
    emojis = extract_emojis(text)
    tags = []
    for e in emojis:
        for char in e:
            if char in POSITIVE_EMOJIS:  tags.append('EMOJI_POS')
            elif char in NEGATIVE_EMOJIS: tags.append('EMOJI_NEG')
            elif char in NEUTRAL_EMOJIS:  tags.append('EMOJI_NEU')
            elif char in SARCASTIC_EMOJIS:tags.append('EMOJI_SARC')
    # Remove original emojis, append tags
    clean = re.sub(
        r'[\U0001F300-\U0001FFFF\U00002702-\U000027B0]+', '', text
    )
    return clean.strip() + ' ' + ' '.join(tags)

In [4]:
#Slang processor
def normalize_elongation(text):
    """Convert 'amazinggggg' → 'amazing ELONGATED'."""
    def _norm(m):
        base = re.sub(r'(.)\1{2,}', r'\1', m.group())
        return base + ' ELONGATED'
    return re.sub(r'\b\w*(\w)\1{2,}\w*\b', _norm, text)
 
def expand_slang(text):
    words = text.split()
    expanded = []
    i = 0
    while i < len(words):
        two_word = ' '.join(words[i:i+2]).lower()
        if two_word in SOCIAL_SLANG:
            expanded.append(SOCIAL_SLANG[two_word])
            i += 2
        elif words[i].lower() in SOCIAL_SLANG:
            expanded.append(SOCIAL_SLANG[words[i].lower()])
            i += 1
        else:
            expanded.append(words[i])
            i += 1
    return ' '.join(expanded)

In [5]:
#Text processor
def preprocess(text):
    text = str(text)
    text = emoji_to_sentiment_tag(text)   # emojis → tags
    text = expand_slang(text)              # tbh → to be honest
    text = normalize_elongation(text)      # loveeeee → love ELONGATED
    text = text.lower()
    text = re.sub(r'[^a-z\s_]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
 
 
def extract_extra_features(texts):
    """Hand-crafted features to add on top of TF-IDF."""
    records = []
    for text in texts:
        emojis = extract_emojis(str(text))
        emoji_chars = ''.join(emojis)
 
        has_pos_emoji  = int(any(c in POSITIVE_EMOJIS  for c in emoji_chars))
        has_neg_emoji  = int(any(c in NEGATIVE_EMOJIS  for c in emoji_chars))
        has_neu_emoji  = int(any(c in NEUTRAL_EMOJIS   for c in emoji_chars))
        has_sarc_emoji = int(any(c in SARCASTIC_EMOJIS for c in emoji_chars))
 
        elongated_count = len(re.findall(r'\b\w*(\w)\1{2,}\w*\b', str(text)))
        caps_ratio = sum(1 for c in str(text) if c.isupper()) / (len(str(text)) + 1)
 
        records.append({
            'has_pos_emoji':   has_pos_emoji,
            'has_neg_emoji':   has_neg_emoji,
            'has_neu_emoji':   has_neu_emoji,
            'has_sarc_emoji':  has_sarc_emoji,
            'elongated_count': elongated_count,
            'caps_ratio':      caps_ratio * 10,   # scale for MultinomialNB (needs ≥ 0)
        })
    return pd.DataFrame(records).values

In [6]:
# Read file
train_df = pd.read_csv('social_media_sentiment_train.csv', encoding ='latin1')
test_df  = pd.read_csv('social_media_sentiment_test.csv', encoding = 'latin1')

In [7]:
#Print the content of both train and test file
print(f"Train: {train_df.shape}  |  Test: {test_df.shape}")
print("\nLabel distribution (train):")
print(train_df['label'].value_counts())

Train: (10000, 2)  |  Test: (2000, 2)

Label distribution (train):
label
Positive     2930
Negative     2850
Sarcastic    2702
Neutral      1518
Name: count, dtype: int64


In [8]:
# Preprocess texts
train_df['clean_text'] = train_df['text'].apply(preprocess)
test_df['clean_text']  = test_df['text'].apply(preprocess)
 
X_train = train_df['clean_text'].tolist()
y_train = train_df['label'].tolist()
X_test  = test_df['clean_text'].tolist()
y_test  = test_df['label'].tolist()

In [9]:
print("\nTraining model...")
 
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 3),
        max_features=60000,
        sublinear_tf=True,
        analyzer='word',
        token_pattern=r'\b[a-z_][a-z_]+\b',
        min_df=2,
    )),
    ('clf', ComplementNB(alpha=0.1))
])
 
pipeline.fit(X_train, y_train)


Training model...


Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=60000, min_df=2,
                                 ngram_range=(1, 3), sublinear_tf=True,
                                 token_pattern='\\b[a-z_][a-z_]+\\b')),
                ('clf', ComplementNB(alpha=0.1))])

In [10]:
y_pred = pipeline.predict(X_test)
 
acc = accuracy_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred, average='weighted')
 
print(f"\n{'='*50}")
print(f"  Accuracy : {acc:.4f}  ({acc*100:.2f}%)")
print(f"  F1 Score : {f1:.4f}  (weighted)")
print(f"{'='*50}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred,
      target_names=['Negative','Neutral','Positive','Sarcastic']))

# Cross-validation on train
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy')
print(f"\n5-Fold CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")


  Accuracy : 0.9990  (99.90%)
  F1 Score : 0.9990  (weighted)

Classification Report:
              precision    recall  f1-score   support

    Negative       1.00      1.00      1.00       543
     Neutral       1.00      1.00      1.00       402
    Positive       1.00      1.00      1.00       535
   Sarcastic       1.00      1.00      1.00       520

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000


5-Fold CV Accuracy: 0.9983 ± 0.0012


In [11]:
COLORS = {
    'Positive':  '#22c55e',
    'Negative':  '#ef4444',
    'Neutral':   '#64748b',
    'Sarcastic': '#f59e0b',
}
CLASS_ORDER = ['Positive', 'Negative', 'Neutral', 'Sarcastic']
 
fig = plt.figure(figsize=(20, 18))
fig.patch.set_facecolor('#0f172a')
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)
 
title_kw  = dict(color='white', fontsize=13, fontweight='bold', pad=10)
label_kw  = dict(color='#94a3b8', fontsize=10)
tick_kw   = dict(colors='#94a3b8', labelsize=9)

<Figure size 2000x1800 with 0 Axes>

In [12]:
# ── (B) Confusion Matrix ── FIXED ───────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1:])
ax2.set_facecolor('#1e293b')
cm = confusion_matrix(y_test, y_pred, labels=CLASS_ORDER)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

im = ax2.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1, aspect='auto')
ax2.set_xticks(range(4)); ax2.set_yticks(range(4))
ax2.set_xticklabels(CLASS_ORDER, color='#94a3b8', fontsize=10)
ax2.set_yticklabels(CLASS_ORDER, color='#94a3b8', fontsize=10)
ax2.set_xlabel('Predicted', color='#94a3b8', fontsize=11)
ax2.set_ylabel('Actual',    color='#94a3b8', fontsize=11)
ax2.set_title('Confusion Matrix (Normalised)', **title_kw)
for i in range(4):
    for j in range(4):
        ax2.text(j, i, f'{cm_norm[i,j]:.2f}\n({cm[i,j]})',
                 ha='center', va='center', fontsize=10,
                 color='white' if cm_norm[i,j] > 0.5 else '#1e293b')

# ✅ KEY FIX: use fig.colorbar with explicit fraction/pad instead of plt.colorbar
cbar = fig.colorbar(im, ax=ax2, fraction=0.03, pad=0.04)
cbar.ax.yaxis.set_tick_params(color='#94a3b8')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='#94a3b8')

[None, None, None, None, None, None, None, None, None, None, None, None]

In [13]:
ax3 = fig.add_subplot(gs[1, 0])
ax3.set_facecolor('#1e293b')
from sklearn.metrics import precision_recall_fscore_support
p, r, f, _ = precision_recall_fscore_support(y_test, y_pred, labels=CLASS_ORDER)
x = np.arange(4)
w = 0.25
ax3.bar(x - w, p, w, label='Precision', color='#60a5fa', edgecolor='none')
ax3.bar(x,     r, w, label='Recall',    color='#34d399', edgecolor='none')
ax3.bar(x + w, f, w, label='F1',        color='#f472b6', edgecolor='none')
ax3.set_xticks(x); ax3.set_xticklabels(CLASS_ORDER, fontsize=9, color='#94a3b8')
ax3.set_ylim(0, 1.1)
ax3.set_title('Per-Class Metrics', **title_kw)
ax3.legend(fontsize=8, facecolor='#0f172a', labelcolor='white', framealpha=0.7)
ax3.tick_params(**tick_kw)
ax3.spines[:].set_visible(False)

In [14]:
ax4 = fig.add_subplot(gs[1, 1])
ax4.set_facecolor('#1e293b')
folds = [f'Fold {i+1}' for i in range(5)]
ax4.bar(folds, cv_scores, color='#818cf8', edgecolor='none', width=0.5)
ax4.axhline(cv_scores.mean(), color='#f472b6', linewidth=2,
            linestyle='--', label=f'Mean={cv_scores.mean():.3f}')
ax4.set_ylim(cv_scores.min() - 0.02, min(1.0, cv_scores.max() + 0.02))
ax4.set_title('5-Fold CV Accuracy', **title_kw)
ax4.legend(fontsize=9, facecolor='#0f172a', labelcolor='white')
ax4.tick_params(**tick_kw)
ax4.spines[:].set_visible(False)

In [15]:
feature_names = pipeline.named_steps['tfidf'].get_feature_names_out()
log_probs     = pipeline.named_steps['clf'].feature_log_prob_
classes       = pipeline.classes_
 
axes_feat = [fig.add_subplot(gs[1, 2]),
             fig.add_subplot(gs[2, 0]),
             fig.add_subplot(gs[2, 1]),
             fig.add_subplot(gs[2, 2])]
 
for ax, cls in zip(axes_feat, CLASS_ORDER):
    ax.set_facecolor('#1e293b')
    idx = list(classes).index(cls)
    top_idx = np.argsort(log_probs[idx])[-10:][::-1]
    top_features = feature_names[top_idx]
    top_scores   = log_probs[idx][top_idx]
    # normalise to 0-1 for display
    norm_scores = (top_scores - top_scores.min()) / (top_scores.max() - top_scores.min() + 1e-9)
    ax.barh(range(10), norm_scores[::-1], color=COLORS[cls], edgecolor='none')
    ax.set_yticks(range(10))
    ax.set_yticklabels(top_features[::-1], fontsize=8, color='white')
    ax.set_title(f'Top Features — {cls}', **title_kw)
    ax.tick_params(axis='x', **tick_kw)
    ax.spines[:].set_visible(False)
 
# ── Overall title + metrics banner ─────────────────────────────────────────
fig.suptitle(
    f'Naive Bayes Sentiment Analysis  |  Accuracy: {acc*100:.1f}%  |  F1: {f1:.3f}',
    color='white', fontsize=16, fontweight='bold', y=0.98
)
 
plt.savefig('naive_bayes_sentiment_results.png',
            dpi=150, bbox_inches='tight', facecolor='#0f172a')
plt.close()
print("\nPlot saved.")


Plot saved.


In [16]:
def predict_sentiment(texts, model=pipeline):
    cleaned = [preprocess(t) for t in texts]
    preds   = model.predict(cleaned)
    probas  = model.predict_proba(cleaned)
    results = []
    for text, pred, prob in zip(texts, preds, probas):
        class_probs = dict(zip(model.classes_, prob.round(3)))
        results.append({
            'text':       text,
            'prediction': pred,
            'confidence': round(max(prob), 3),
            **class_probs
        })
    return pd.DataFrame(results)
 
demo_texts = [
    "tbh thissss is lit 🔥",
    "omg it was normallll 🤷",
    "ngl this is trash 😡",
    "oh yeah totally great 🙄 best experience ever 👏",
    "highkey this is amazing ✨",
    "lowkey it was fineeeee 🫤",
]
 
print("\n── Live Inference Demo ──────────────────────────────────────────────────")
results = predict_sentiment(demo_texts)
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.float_format', '{:.3f}'.format)
print(results[['text','prediction','confidence']].to_string(index=False))


── Live Inference Demo ──────────────────────────────────────────────────
                                          text prediction  confidence
                          tbh thissss is lit 🔥   Positive       0.981
                        omg it was normallll 🤷    Neutral       1.000
                           ngl this is trash 😡   Negative       0.997
oh yeah totally great 🙄 best experience ever 👏  Sarcastic       1.000
                     highkey this is amazing ✨   Positive       0.996
                      lowkey it was fineeeee 🫤    Neutral       1.000
